|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The map<h1>|
|<h2>Lecture:</h2>|<h1><b>The map, in words that you know<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

# You already know this shape

A web service has a shape that is in your fingers by now: a router, a
service, a repo, a database. The models and the migrations live in their own
files. When a request fails, the shape tells you where to look first.

An **inference** engine has a shape too. It is the server that runs a trained
**model** (a large function that predicts the next piece of text) and answers
requests with it. Most courses give it to you at the end,
after you build the parts. This course gives it to you **first**. Every stage
after this notebook builds one box of it, and each stage guide starts with
"YOU ARE HERE".

This notebook needs no **GPU** (the processor on the graphics card, which
runs the same arithmetic on much data at once) and has no code. Read it slowly. The same map is
on one page in `docs/MAP.md`. Keep that page open while you work.

# The words first

GPU people use many words that sound hard. Most of them are new names for
ideas that you already know. Here is the idea first, and then the name.

| The idea, in software words | The name that GPU people use |
|---|---|
| A function. Text goes in, and a score for each possible next piece of text comes out. | the **model** |
| The piece of text that the model reads and writes: a word or a part of a word, as an integer id. | a **token** |
| A large read-only table of numbers. The model reads all of it to make one token. | the **weights** |
| The requests that the engine serves in the same call. | the **batch** |
| The first call for a request. It reads the whole prompt at once. | the **prefill** |
| Each call after that. It makes one new token for each request. | a **decode** step |
| A memo table for each request. The model keeps its work on the earlier tokens there, so that it never does that work again. | the **KV cache** |
| A fixed-size page of that memo table. | a **block** |
| A function that runs on the GPU. | a **kernel** |
| A call from the CPU that starts a kernel. It has a fixed cost, like an RPC. | a **kernel launch** |
| The main memory of the GPU. | **HBM** |
| Code that waits for memory reads, not for its arithmetic. The GPU version of "I/O-bound". | **memory-bound** |
| Evict a request from memory, and run it again later. | **preemption** |
| A slow version of the code that you trust. You compare the fast version with it. | an **oracle** |

Three of these carry a real idea that is new to you: the KV cache,
memory-bound, and the batch. The course spends most of its time on those
three. The other names are only labels.

# The layers

Here is the engine, in the words of a web service. The left column is the
word that you know. Each box is a part that you will build.

```
  client ── POST /v1/chat/completions
     │
┌────▼──────────────────────────────────────────────────────────┐
│ ROUTER      the API server: parse, stream, stop on disconnect │
├───────────────────────────────────────────────────────────────┤
│ SERVICE     the engine loop, one step after another           │
│             ├── the scheduler: who runs in this step?         │
│             ├── the sampler: scores → one token for each row  │
│             └── the detokenizer: tokens → text                │
├───────────────────────────────────────────────────────────────┤
│ REPO        the KV cache manager: the memo tables of all the  │
│             requests, in fixed-size pages                     │
├───────────────────────────────────────────────────────────────┤
│ DB DRIVER   the model runner: one flat batch for each step    │
├───────────────────────────────────────────────────────────────┤
│ DB ENGINE   the model, and the GPU functions under it         │
└───────────────────────────────────────────────────────────────┘
  SCHEMA      the model config: it sets the size of everything
```

The analogy is not exact. `docs/MAP.md` has a table of each pattern, with the
nearest software pattern and the point where the analogy stops. That point is
usually the lesson of the stage.

# The life of one request

Follow one chat request through the boxes. Each step names the box.

1. **Router.** The request arrives. The server checks the limits and puts an
   "add this request" command in a queue.
2. **Service.** The engine loop takes the command at the start of its next
   step. It turns the prompt into tokens and puts the request in the waiting
   queue.
3. **Service → Repo.** The scheduler plans the step. The running requests get
   one new token each. Waiting requests join, until the step is full. The repo
   gives each request the pages of memo table that it needs.
4. **Repo is full.** No free pages? The engine evicts a request, and runs it
   again later.
5. **DB driver.** The runner puts all the requests of the step into one flat
   batch.
6. **DB engine.** The model runs one step for the whole batch. It reads all
   its weights one time, and it reads the memo table of each request.
7. **Service.** The **sampler** picks one token for each request from the
   scores. The **detokenizer** turns the new tokens into text.
8. **Router.** The new text goes to the client as a stream.
9. **The end.** The request finishes, or the client leaves. The repo takes
   its pages back.

Steps 3 to 8 are **one step** of the engine. They repeat, for all the running
requests together, until nothing is left.

### What to remember from this notebook

- The engine has five layers, and you know their shape: router, service, repo,
  DB driver, DB engine. The model config is the schema.
- One step serves every running request together. This one fact is the source
  of most of the surprises in this course.
- When something breaks, the first question is the same as in your web
  services: **which layer?** The challenge of this section trains that.

The shape of your instinct transfers. Some of the assumptions under it do
not. The next notebook finds them, and you measure each one.